## Ebay API Sandbox
- An area for testing
- General Docs: https://developer.ebay.com/develop/get-started
- Requests tutorial: https://www.geeksforgeeks.org/python/python-requests-tutorial/
- Close example: https://www.youtube.com/watch?v=flbuoG9rAs4

In [1]:
from dotenv import load_dotenv
import os
import requests
import base64
import polars as pl

In [2]:
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

APP_ID = os.getenv('EBAY_CLIENT_ID')

In [3]:

# Use requests to get a response. 200 means success
url = "https://ebay.com"
response = requests.get(url)
print(response.status_code)

200


In [4]:

client_id = os.getenv('EBAY_CLIENT_ID')
client_secret = os.getenv('EBAY_SECRET')

def get_ebay_token():
    auth_string = f"{client_id}:{client_secret}"
    b64_encoded = base64.b64encode(auth_string.encode()).decode()
    
    url = 'https://api.ebay.com/identity/v1/oauth2/token' # Use api.sandbox.ebay.com for testing
    headers = {
        'Authorization': f'Basic {b64_encoded}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'client_credentials',
        'scope': 'https://api.ebay.com/oauth/api_scope'
    }
    
    response = requests.post(url, headers=headers, data=data)
    response_data = response.json()
    
    return response_data.get('access_token')

# Get the token to use in subsequent requests
token = get_ebay_token()


In [5]:
def search_ebay_items(access_token, query):
    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    params = {
        'q': query,
        'limit': 5
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        return f"Error {response.status_code}: {response.text}"

results = search_ebay_items(token, "2025 POKEMON JAPANESE M-P PROMO MCDONALD'S #020 PIKACHU PSA 10")
print(results)


{'href': 'https://api.ebay.com/buy/browse/v1/item_summary/search?q=2025+POKEMON+JAPANESE+M-P+PROMO+MCDONALD%27S+%23020+PIKACHU+PSA+10&limit=5&offset=0', 'total': 1109, 'next': 'https://api.ebay.com/buy/browse/v1/item_summary/search?q=2025+POKEMON+JAPANESE+M-P+PROMO+MCDONALD%27S+%23020+PIKACHU+PSA+10&limit=5&offset=5', 'limit': 5, 'offset': 0, 'itemSummaries': [{'itemId': 'v1|137514925750|0', 'title': "POKEMON CARD 2025 US SELLER PSA 10 McDonald's Pikachu 020/M-P Japanese Promo", 'leafCategoryIds': ['2552'], 'categories': [{'categoryId': '2552', 'categoryName': 'Other Card Games & Poker'}, {'categoryId': '220', 'categoryName': 'Toys & Hobbies'}, {'categoryId': '233', 'categoryName': 'Games'}, {'categoryId': '180350', 'categoryName': 'Card Games & Poker'}], 'image': {'imageUrl': 'https://i.ebayimg.com/images/g/T6EAAeSwM~Rp~UCM/s-l225.jpg'}, 'price': {'value': '56.83', 'currency': 'USD'}, 'itemHref': 'https://api.ebay.com/buy/browse/v1/item/v1%7C137514925750%7C0', 'seller': {'username': '

In [6]:

def items_to_df(results: dict) -> pl.DataFrame:
    rows = []
    for item in results.get("itemSummaries", []):
        price = item.get("price") or {}
        seller = item.get("seller") or {}
        rows.append({
            "item_id": item.get("itemId"),
            "title": item.get("title"),
            "condition": item.get("condition"),
            "price": float(price["value"]) if price.get("value") is not None else None,
            "currency": price.get("currency"),
            "seller_feedback_pct": seller.get("feedbackPercentage"),
            "item_url": item.get("itemWebUrl"),
        })
    return pl.DataFrame(rows)

df = items_to_df(results)